# 🧠 Módulo 03 — Treinamento com RSL-RL + PPO

**Duração:** 35 minutos (1:25 – 2:00) | **Workshop Quadrúpede — Summit IA Joinville**

### 🎯 Objetivos
1. ✅ Entender o **RSL-RL** e por que usamos ele
2. ✅ Compreender o algoritmo **PPO** de forma intuitiva
3. ✅ Conhecer a **arquitetura de rede neural** (Actor-Critic MLP)
4. ✅ Analisar os **hiperparâmetros** de treinamento
5. ✅ **Lançar o treinamento** no servidor GPU
6. ✅ Monitorar com **TensorBoard**


## 📚 O que é o RSL-RL?

**RSL-RL** (*Robotic Systems Lab Reinforcement Learning*) é uma biblioteca de RL desenvolvida pelo **ETH Zurich** — o mesmo grupo que criou o Anymal C.

| Característica | Detalhes |
|---|---|
| **Algoritmo** | PPO (Proximal Policy Optimization) |
| **Otimizado para** | Ambientes paralelos em GPU |
| **Arquitetura** | Actor-Critic MLP separados |
| **Batch** | 100% em GPU (sem transferência CPU↔GPU) |
| **Paper** | ANYmal locomotion (Kumar et al., 2021) |

```
RSL-RL vs outros:
  RSL-RL:  ★ Mais rápido para GPU-parallel (Isaac Lab padrão)
  skrl:    ★ Mais algoritmos (PPO, SAC, DDPG...)
  SB3:     ✗ Não otimizado para GPU-parallel (mais lento)
```


## 🧠 PPO — Proximal Policy Optimization

### 💡 Intuição:

```
Tentativa 1: Política P₁ → Recompensa R₁

'Vou fazer mais do que funcionou — mas com mudança gradual!'

Tentativa 2: P₂ ≈ P₁ (mas um pouco melhor)
Tentativa N: 🐾 Robô andando bem!

PPO garante: não mais que ε=20% de mudança por iteração
```

### 🔑 Conceitos-chave:

**1. Actor-Critic:**
```
   obs(48D)
      ├──▶ ACTOR  (512→256→128→12)  → ação: 'O que fazer'
      └──▶ CRITIC (512→256→128→1)   → V(s): 'Quão bom é este estado'
```

**2. PPO Clipping:**
```python
r(θ) = π_novo(a|s) / π_antigo(a|s)  # ratio de probabilidades
L = min(r(θ)·A, clip(r(θ), 1-0.2, 1+0.2)·A)  # ε = 0.2
# A = vantagem: reward - baseline (quão surpreendente foi a ação)
```

**3. GAE (Vantagem):**
```
A(s,a) > 0 → ação foi MELHOR que esperado → faça mais
A(s,a) < 0 → ação foi PIOR que esperado  → faça menos
```


## 🏗️ Arquitetura de Rede Neural

```
ACTOR (Política π):
  obs(48) → Linear(48→512) → ELU
         → Linear(512→256) → ELU
         → Linear(256→128) → ELU
         → Linear(128→12) → Tanh
         → ação normalizada ∈ [-1, 1]¹²
  Parâmetros: ~432K

CRITIC (Value function V):
  obs(48) → Linear(48→512) → ELU
         → Linear(512→256) → ELU
         → Linear(256→128) → ELU
         → Linear(128→1)
         → V(s) escalar
  Parâmetros: ~432K

Total: ~864K parâmetros (pequeno — treina rápido!)
```

**Por que ELU?** Mais suave que ReLU, evita 'neurônios mortos', melhor para física
**Por que Tanh na saída?** Garante ações no range [-1, 1]


In [ ]:
# 📄 Mostrar configuração PPO
try:
    from workshop_quadrupede.agents import AnymalWorkshopPPORunnerCfg
    cfg = AnymalWorkshopPPORunnerCfg()
    print('✅ AnymalWorkshopPPORunnerCfg carregada!')
    print(f'  max_iterations:      {cfg.max_iterations}')
    print(f'  num_steps_per_env:   {cfg.num_steps_per_env}')
    print(f'  actor_hidden_dims:   {cfg.policy.actor_hidden_dims}')
    print(f'  learning_rate:       {cfg.algorithm.learning_rate}')
    print(f'  clip_param:          {cfg.algorithm.clip_param}')
    print(f'  gamma:               {cfg.algorithm.gamma}')
    print(f'  num_learning_epochs: {cfg.algorithm.num_learning_epochs}')
    num_envs = 4096
    total_steps = cfg.max_iterations * cfg.num_steps_per_env * num_envs
    print(f'\n  Total steps ({num_envs} envs): {total_steps:,}')
except ImportError:
    print('⚠️  Pacote não disponível neste kernel.')
    print('   Configuração padrão:')
    print('  max_iterations:      1500')
    print('  num_steps_per_env:   24')
    print('  actor_hidden_dims:   [512, 256, 128]')
    print('  learning_rate:       0.001')
    print('  clip_param:          0.2')
    print('  gamma:               0.99')
    print('  num_learning_epochs: 5')
    print('  Total steps: 147,456,000')


## 📊 Hiperparâmetros Explicados

| Hiperparâmetro | Valor | Impacto se mudar |
|---|---|---|
| `num_steps_per_env` | 24 | ↑ mais estável, ↓ mais rápido |
| `num_envs` | 4096 | ↑ mais rápido, ↑ VRAM |
| `learning_rate` | 1e-3 | ↑ instável, ↓ lento |
| `clip_param` | 0.2 | ↑ instável, ↓ conservador |
| `gamma` | 0.99 | ↑ pensa mais no futuro |
| `entropy_coef` | 0.005 | ↑ mais exploração |
| `num_learning_epochs` | 5 | ↑ sample-efficient, ↑ overfitting |
| `desired_kl` | 0.01 | Controla velocidade de mudança |

### 🎯 Dicas de tuning:
- **Não converge rápido?** → `num_envs=8192` ou `learning_rate=3e-3`
- **Instável?** → `clip_param=0.1` ou `learning_rate=5e-4`
- **VRAM estourando?** → `num_envs=2048`


## 🚀 Lançando o Treinamento

```bash
# Treinamento padrão (4096 envs, ~45min RTX 4090):
cd /IsaacLabTutorial
./isaaclab.sh -p workshop/scripts/treinar.py \
    --task Workshop-Anymal-v0 \
    --num_envs 4096 \
    --headless

# Com menos VRAM (RTX 3090 com 24 GB):
/isaac-sim/python.sh workshop/scripts/treinar.py \
    --task Workshop-Anymal-v0 \
    --num_envs 2048 \
    --headless
```

### 📁 Saída gerada:
```
logs/workshop_quadrupede_anymal/
  └── 2026-05-22_14-30-00/
      ├── model_100.pt   ← checkpoint iter 100
      ├── model_500.pt
      ├── model_1500.pt  ← checkpoint final
      └── events.out.tfevents.*  ← TensorBoard logs
```

| GPU | Num Envs | Tempo estimado |
|-----|----------|----------------|
| RTX 3090 | 4096 | ~50-60 min |
| RTX 4090 | 4096 | ~30-45 min |
| A100 | 4096 | ~20-30 min |


In [ ]:
# 🚀 Exibir e copiar comando de treinamento
print('=' * 65)
print('🚀 COPIE E COLE ESTE COMANDO NO TERMINAL:')
print('=' * 65)
print()
print('cd /IsaacLabTutorial')
print('./isaaclab.sh -p workshop/scripts/treinar.py \\')
print('    --task Workshop-Anymal-v0 \\')
print('    --num_envs 4096 \\')
print('    --headless')
print()
print('=' * 65)
print()

# Calcular estatísticas
num_envs = 4096
steps_per_env = 24
iters = 1500
total = num_envs * steps_per_env * iters
print(f'📊 Estatísticas do treinamento:')
print(f'   Amostras/iteração: {num_envs * steps_per_env:,}')
print(f'   Total de steps:    {total:,}')
print(f'   Checkpoints:       a cada 100 iters')


In [ ]:
# 📊 Configurar e lançar TensorBoard
import os, socket

log_dir = os.path.join(os.getcwd(), 'logs', 'workshop_quadrupede_anymal')
os.makedirs(log_dir, exist_ok=True)

def porta_livre(p):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('localhost', p)) != 0

print(f'📁 Pasta de logs: {log_dir}')
print(f'📊 Porta 6006 disponível: {porta_livre(6006)}')
print()
print('Para lançar TensorBoard:')
print(f'  tensorboard --logdir={log_dir} --port=6006 --host=0.0.0.0 &')
print()
print('Acesse: http://localhost:6006  (ou http://<IP>:6006)')


## 📊 O que Monitorar no TensorBoard

| Métrica | Sinal de sucesso |
|---------|-----------------|
| `Train/mean_reward` | ↑ Subindo → robô aprendendo! |
| `Train/mean_episode_length` | ↑ Subindo → robô não cai mais |
| `Loss/value_function` | ↓ Diminuindo → Critic aprendendo |
| `Loss/entropy` | Estável, não zero → política ainda explora |

### ✅ Treinamento indo bem:
```
iter   0-100: reward negativo, robô cai rapidamente
iter 100-400: reward subindo, primeiros passos
iter 400-800: marcha emergindo, episódios mais longos
iter 800-1500: convergência, robô andando bem
```

### ⏰ Dica de Workshop:
Treinamento completo = ~45 min. Para **não perder tempo**, usaremos um **checkpoint pré-treinado** no Módulo 04!

---

### ➡️ Próximo: Módulo 04 — Resultados e Próximos Passos
